In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Entradas e saídas dos primitivos

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** A versão beta de um novo modelo de execução já está disponível. O modelo de execução dirigida oferece mais flexibilidade ao personalizar seu fluxo de trabalho de mitigação de erros. Consulte o guia [Modelo de execução dirigida](/guides/directed-execution-model) para mais informações.


<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Esta página apresenta uma visão geral das entradas e saídas dos primitivos do Qiskit Runtime que executam cargas de trabalho nos recursos de computação IBM Quantum&reg;. Esses primitivos permitem que você defina eficientemente cargas de trabalho vetorizadas usando uma estrutura de dados conhecida como **Primitive Unified Bloc (PUB)**. Esses PUBs são a unidade fundamental de trabalho que uma QPU precisa para executar essas cargas de trabalho. Eles são usados como entradas para o método [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) dos primitivos Sampler e Estimator, que executam a carga de trabalho definida como um job. Em seguida, após a conclusão do job, os resultados são retornados em um formato que depende tanto dos PUBs utilizados quanto das opções de runtime especificadas pelos primitivos Sampler ou Estimator.
<span id="pubs"></span>
## Visão geral dos PUBs
Ao invocar o método [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) de um primitivo, o principal argumento necessário é uma `list` de uma ou mais tuplas — uma para cada Circuit sendo executado pelo primitivo. Cada uma dessas tuplas é considerada um PUB, e os elementos obrigatórios de cada tupla na lista dependem do primitivo utilizado. Os dados fornecidos a essas tuplas também podem ser organizados em diversas formas para oferecer flexibilidade em uma carga de trabalho por meio de broadcasting — cujas regras são descritas em uma [seção seguinte](#broadcasting-rules).

### PUB do Estimator
Para o primitivo Estimator, o formato do PUB deve conter no máximo quatro valores:
- Um único `QuantumCircuit`, que pode conter um ou mais objetos [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Uma lista de um ou mais observáveis, que especificam os valores esperados a estimar, organizados em um array (por exemplo, um único observável representado como um array 0-d, uma lista de observáveis como um array 1-d, e assim por diante). Os dados podem estar em qualquer um dos formatos `ObservablesArrayLike`, como `Pauli`, `SparsePauliOp`, `PauliList` ou `str`.
   > **Note:** Se você tiver dois observáveis comutativos em PUBs diferentes, mas com o mesmo Circuit, eles não serão estimados usando a mesma medição. Cada PUB representa uma base diferente para medição e, portanto, medições separadas são necessárias para cada PUB. Para garantir que observáveis comutativos sejam estimados usando a mesma medição, eles devem ser agrupados dentro do mesmo PUB.
- Uma coleção de valores de parâmetros para vincular ao Circuit. Isso pode ser especificado como um único objeto semelhante a array, onde o último índice é sobre os objetos `Parameter` do Circuit, ou omitido (ou equivalentemente, definido como `None`) se o Circuit não tiver objetos `Parameter`.
- (Opcionalmente) uma precisão alvo para os valores esperados a estimar

### PUB do Sampler
Para o primitivo Sampler, o formato da tupla PUB contém no máximo três valores:
- Um único `QuantumCircuit`, que pode conter um ou mais objetos [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
   *Nota: Esses Circuits também devem incluir instruções de medição para cada um dos qubits a serem amostrados.*
- Uma coleção de valores de parâmetros para vincular ao Circuit $\theta_k$ (necessário somente se algum objeto `Parameter` for usado e precisar ser vinculado em tempo de execução)
- (Opcionalmente) um número de shots para medir o Circuit
---

O código a seguir demonstra um conjunto de exemplo de entradas vetorizadas para o primitivo `Estimator` e as executa em um backend IBM&reg; como um único objeto `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV2 as Sampler,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

### Regras de broadcasting
Os PUBs agregam elementos de múltiplos arrays (observáveis e valores de parâmetros) seguindo as mesmas regras de broadcasting do NumPy. Esta seção resume brevemente essas regras. Para uma explicação detalhada, consulte a [documentação de regras de broadcasting do NumPy](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Regras:

* Arrays de entrada não precisam ter o mesmo número de dimensões.
  * O array resultante terá o mesmo número de dimensões que o array de entrada com a maior dimensão.
  * O tamanho de cada dimensão é o maior tamanho da dimensão correspondente.
  * Dimensões ausentes são assumidas como tendo tamanho um.
* As comparações de forma começam pela dimensão mais à direita e continuam para a esquerda.
* Duas dimensões são compatíveis se seus tamanhos forem iguais ou se um deles for 1.

Exemplos de pares de arrays que fazem broadcasting:

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

Exemplos de pares de arrays que não fazem broadcasting:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

`EstimatorV2` retorna uma estimativa de valor esperado para cada elemento da forma com broadcasting aplicado.

Aqui estão alguns exemplos de padrões comuns expressos em termos de broadcasting de arrays. Sua representação visual correspondente é mostrada na figura a seguir:

Os conjuntos de valores de parâmetros são representados por arrays n x m, e os arrays de observáveis são representados por uma ou mais arrays de coluna única. Para cada exemplo no código anterior, os conjuntos de valores de parâmetros são combinados com seu array de observáveis para criar as estimativas de valores esperados resultantes.

 - *Exemplo 1*: (broadcasting de observável único) tem um conjunto de valores de parâmetros que é um array 5x1 e um array de observáveis 1x1. O único item no array de observáveis é combinado com cada item no conjunto de valores de parâmetros para criar um único array 5x1, onde cada item é uma combinação do item original no conjunto de valores de parâmetros com o item no array de observáveis.

 - *Exemplo 2*: (zip) tem um conjunto de valores de parâmetros 5x1 e um array de observáveis 5x1. A saída é um array 5x1, onde cada item é uma combinação do n-ésimo item no conjunto de valores de parâmetros com o n-ésimo item no array de observáveis.

  - *Exemplo 3*: (produto externo/cartesiano) tem um conjunto de valores de parâmetros 1x6 e um array de observáveis 4x1. Sua combinação resulta em um array 4x6 criado combinando cada item no conjunto de valores de parâmetros com *cada* item no array de observáveis, e assim cada valor de parâmetro se torna uma coluna inteira na saída.

  - *Exemplo 4*: (generalização nd padrão) tem um array de conjunto de valores de parâmetros 3x6 e dois arrays de observáveis 3x1. Esses se combinam para criar dois arrays de saída 3x6 de forma semelhante ao exemplo anterior.

![Esta imagem ilustra várias representações visuais de broadcasting de arrays](../docs/images/guides/primitive-input-output/broadcasting.svg "Representação visual de broadcasting")

In [4]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  3 254]
 [  0   0]
 [  3 255]
 ...
 [  0   0]
 [  3 255]
 [  0   0]]



<Admonition type="tip" title="SparsePauliOp">
Cada `SparsePauliOp` conta como um único elemento neste contexto, independentemente do número de Paulis contidos no `SparsePauliOp`. Portanto, para fins dessas regras de broadcasting, todos os seguintes elementos têm a mesma forma:

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111110': 199, '0000000000': 1337, '1111111111': 1052, '1111111000': 33, '1110000000': 65, '1100100000': 2, '1100000000': 25, '0010001110': 1, '0000000011': 30, '1111111011': 58, '1111111010': 25, '0000000110': 7, '0010000001': 11, '0000000001': 179, '1110111110': 6, '1111110000': 33, '1111101111': 49, '1110111111': 40, '0000111010': 2, '0100000000': 35, '0000000010': 51, '0000100000': 31, '0110000000': 7, '0000001111': 22, '1111111100': 24, '1011111110': 5, '0001111111': 58, '0000111111': 24, '1111101110': 10, '0000010001': 5, '0000001001': 2, '0011111111': 38, '0000001000': 11, '1111100000': 34, '0111111111': 45, '0000000100': 18, '0000000101': 2, '1011111111': 11, '1110000001': 13, '1101111000': 1, '0010000000': 52, '0000010000': 17, '0000011111': 15, '1110100001': 1, '0111111110': 9, '0000000111': 19, '1101111111': 15, '1111110111': 17, '0011111110': 5, '0001101110': 1, '0111111011': 6, '0100001000': 2, '0010001111': 1, '1111011000': 1, '0000111110': 4, '0011110010': 1

As seguintes listas de operadores, embora equivalentes em termos de informação contida, têm formas diferentes:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


</Admonition>
## Visão geral das saídas dos primitivos
Depois que um ou mais PUBs são enviados a uma QPU para execução e um job é concluído com sucesso, os dados são retornados como um objeto contêiner [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), acessado chamando o método `RuntimeJobV2.result()`. O `PrimitiveResult` contém uma lista iterável de objetos [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) que contêm os resultados de execução para cada PUB. Dependendo do primitivo usado, esses dados serão valores esperados e suas barras de erro no caso do Estimator, ou amostras da saída do Circuit no caso do Sampler.

Cada elemento desta lista corresponde a cada PUB submetido ao método `run()` do primitivo (por exemplo, um job submetido com 20 PUBs retornará um objeto `PrimitiveResult` que contém uma lista de 20 `PubResults`, um correspondendo a cada PUB).

Cada um desses objetos `PubResult` possui atributos `data` e `metadata`. O atributo `data` é um [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personalizado que contém os valores de medição reais, desvios padrão e assim por diante. Este `DataBin` tem vários atributos dependendo da forma ou estrutura do PUB associado, bem como das opções de mitigação de erros especificadas pelo primitivo usado para submeter o job (por exemplo, [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) ou [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)). Enquanto isso, o atributo `metadata` contém informações sobre as opções de runtime e mitigação de erros usadas (explicado posteriormente na seção [Metadados do resultado](#result-metadata) desta página).

A seguir está um esboço visual da estrutura de dados `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Saída do Estimator">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Saída do Sampler">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

Em termos simples, um único job retorna um objeto `PrimitiveResult` e contém uma lista de um ou mais objetos `PubResult`. Esses objetos `PubResult` armazenam os dados de medição de cada PUB que foi submetido ao job.

Cada `PubResult` possui formatos e atributos diferentes com base no tipo de primitivo que foi usado para o job. Os detalhes são explicados abaixo.

### Saída do Estimator
Cada `PubResult` para o primitivo Estimator contém pelo menos um array de valores esperados (`PubResult.data.evs`) e desvios padrão associados (seja `PubResult.data.stds` ou `PubResult.data.ensemble_standard_error`, dependendo do `resilience_level` usado), mas pode conter mais dados dependendo das opções de mitigação de erros especificadas.

O trecho de código abaixo descreve o formato `PrimitiveResult` (e o `PubResult` associado) para o job criado acima.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  0 135]
 [  0 247]
 [  1 247]
 ...
 [  0   0]
 [  1 224]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0 135]
 [  0 247]
 [  1 247]
 [  1 128]
 [  1 255]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.068359375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.06396484375

The shape of the merged results is (4096, 2).
The bytes of the merged results:
[[  1  15]
 [  1 239]
 [  3 239]
 

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'execution' : {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 08:07:33', stop='2026-01-15 08:07:36', size=4096>)])},
'version' : 2,

The metadata of the PubResult result is:
'circuit_metadata' : {},


#### Como o Estimator calcula o erro
Além da estimativa da média dos observáveis passados nos PUBs de entrada (o campo `evs` do `DataBin`), o Estimator também tenta fornecer uma estimativa do erro associado a esses valores esperados. Todas as consultas do estimador irão preencher o campo `stds` com uma quantidade semelhante ao erro padrão da média para cada valor esperado, mas algumas opções de mitigação de erros produzem informações adicionais, como `ensemble_standard_error`.

Considere um único observável $\mathcal{O}$. Na ausência de [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), você pode pensar em cada shot da execução do Estimator como fornecendo uma estimativa pontual do valor esperado $\langle \mathcal{O} \rangle$. Se as estimativas pontuais estiverem em um vetor `Os`, então o valor retornado em `ensemble_standard_error` é equivalente ao seguinte (no qual $\sigma_{\mathcal{O}}$ é o [desvio padrão da estimativa do valor esperado](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) e $N_{shots}$ é o número de shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

que trata todos os shots como parte de um único ensemble. Se você solicitou [twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling) de gates (`twirling.enable_gates = True`), você pode classificar as estimativas pontuais de $\langle \mathcal{O} \rangle$ em conjuntos que compartilham um twirl comum. Chame esses conjuntos de estimativas de `O_twirls`, e há `num_randomizations` (número de twirls) deles. Então `stds` é o erro padrão da média de `O_twirls`, como em

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

onde $\sigma_{\mathcal{O}}$ é o desvio padrão de `O_twirls` e $N_{twirls}$ é o número de twirls. Quando você não ativa o twirling, `stds` e `ensemble_standard_error` são iguais.

Se você ativar o ZNE, então os `stds` descritos acima se tornam pesos em uma regressão não linear a um modelo extrator. O que finalmente é retornado nos `stds` nesse caso é a incerteza do modelo ajustado avaliado em um fator de ruído igual a zero. Quando há um ajuste ruim, ou grande incerteza no ajuste, os `stds` relatados podem se tornar muito grandes. Quando o ZNE está ativado, `pub_result.data.evs_noise_factors` e `pub_result.data.stds_noise_factors` também são preenchidos, para que você possa fazer sua própria extrapolação.
### Saída do Sampler
Quando um job do Sampler é concluído com sucesso, o objeto [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) retornado contém uma lista de [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult)s, um por PUB. Os data bins desses objetos `SamplerPubResult` são objetos semelhantes a dicionários que contêm um `BitArray` por `ClassicalRegister` no Circuit.

A classe `BitArray` é um contêiner para dados de shots ordenados. Em mais detalhes, ela armazena as bitstrings amostradas como bytes dentro de um array bidimensional. O eixo mais à esquerda deste array percorre os shots ordenados, enquanto o eixo mais à direita percorre os bytes.

Como primeiro exemplo, vamos examinar o seguinte Circuit de dez qubits: